# beniget — Python Def-Use / Use-Def Chain Analysis
Install: `pip install beniget`

In [1]:
import beniget
print(dir(beniget))

['Ancestors', 'Def', 'DefUseChains', 'UseDefChains', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'beniget', 'ordered_set', 'version']


In [2]:
from beniget import DefUseChains, UseDefChains, Def, Ancestors
print('DefUseChains:', dir(DefUseChains))
print('Def:', dir(Def))
print('Ancestors:', dir(Ancestors))

DefUseChains: ['DefinitionContext', 'ScopeContext', 'SwitchScopeContext', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_dump_locals', '_first_non_comprehension_scope', '_support_stdlib', 'add_to_definition', 'add_to_locals', 'compute_annotation_defs', 'compute_defs', 'defs', 'dump_chains', 'dump_definitions', 'extend_definition', 'extend_global', 'generic_visit', 'invalid_name_lookup', 'is_global', 'is_nonlocal', 'location', 'process_annotations', 'process_body', 'process_functions_bodies', 'process_undefs', 'set_definition', 'set_or_extend_global', 'unbound_identifier', 'visit', 'visit_AnnAssign', 'visit_Assert', 'visit_Assign', 'visit_AsyncFor', 'visit_AsyncFunctionDef', 'v

In [3]:
# Raw def-use chain data for a redditwarp file — full JSON output
import ast, os, json
from beniget import DefUseChains

target = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp', 'exceptions.py')
src = open(target, encoding='utf-8').read()
module = ast.parse(src)

duc = DefUseChains()
duc.visit(module)

output = []
for node, defn in duc.chains.items():
    entry = {
        'node_type': type(node).__name__,
        'lineno': getattr(node, 'lineno', None),
        'col_offset': getattr(node, 'col_offset', None),
        'def_name': defn.name() if callable(defn.name) else str(defn.name),
        'def_islive': defn.islive,
        'user_count': len(defn.users()),
        'users': [
            {
                'node_type': type(u.node).__name__,
                'lineno': getattr(u.node, 'lineno', None),
                'name': u.name() if callable(u.name) else str(u.name)
            }
            for u in defn.users()
        ]
    }
    output.append(entry)

print(json.dumps(output, indent=2))

[
  {
    "node_type": "alias",
    "lineno": 2,
    "col_offset": 23,
    "def_name": "annotations",
    "def_islive": true,
    "user_count": 0,
    "users": []
  },
  {
    "node_type": "alias",
    "lineno": 3,
    "col_offset": 19,
    "def_name": "TYPE_CHECKING",
    "def_islive": true,
    "user_count": 1,
    "users": [
      {
        "node_type": "Name",
        "lineno": 4,
        "name": "TYPE_CHECKING"
      }
    ]
  },
  {
    "node_type": "alias",
    "lineno": 3,
    "col_offset": 34,
    "def_name": "Any",
    "def_islive": true,
    "user_count": 1,
    "users": [
      {
        "node_type": "Name",
        "lineno": 129,
        "name": "Any"
      }
    ]
  },
  {
    "node_type": "alias",
    "lineno": 3,
    "col_offset": 39,
    "def_name": "Mapping",
    "def_islive": true,
    "user_count": 1,
    "users": [
      {
        "node_type": "Name",
        "lineno": 136,
        "name": "Mapping"
      }
    ]
  },
  {
    "node_type": "Name",
    "lineno": 4,
 

In [4]:
# Raw scope-level locals for every scope in redditwarp
import ast, os, json
from beniget import DefUseChains

rw = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp')
output = []

for root, dirs, files in os.walk(rw):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    for fname in files:
        if not fname.endswith('.py'):
            continue
        fpath = os.path.join(root, fname)
        try:
            src = open(fpath, encoding='utf-8', errors='ignore').read()
            module = ast.parse(src)
            duc = DefUseChains()
            duc.visit(module)
            rel = os.path.relpath(fpath, rw)

            for scope_node, defs_set in duc.locals.items():
                scope = type(scope_node).__name__
                scope_name = getattr(scope_node, 'name', '<module>')
                scope_lineno = getattr(scope_node, 'lineno', None)
                for d in defs_set:
                    output.append({
                        'file': rel,
                        'scope_type': scope,
                        'scope_name': scope_name,
                        'scope_lineno': scope_lineno,
                        'def_name': d.name() if callable(d.name) else str(d.name),
                        'def_node_type': type(d.node).__name__,
                        'def_lineno': getattr(d.node, 'lineno', None),
                        'def_islive': d.islive,
                        'user_count': len(d.users()),
                        'is_unused': len(d.users()) == 0
                    })
        except Exception:
            pass

print(json.dumps(output, indent=2))

W: unbound identifier '__class__' at <unknown>:551:38
W: unbound identifier '__class__' at <unknown>:95:41


W: unbound identifier 'x' at <unknown>:299:18
W: unbound identifier 'x' at <unknown>:299:64
W: unbound identifier 'x' at <unknown>:299:18
W: unbound identifier 'x' at <unknown>:299:64
W: unbound identifier 'x' at <unknown>:72:22
W: unbound identifier 'x' at <unknown>:72:68
W: unbound identifier 'x' at <unknown>:72:22
W: unbound identifier 'x' at <unknown>:72:68
W: unbound identifier 'x' at <unknown>:201:24
W: unbound identifier 'x' at <unknown>:201:70
W: unbound identifier 'x' at <unknown>:201:24
W: unbound identifier 'x' at <unknown>:201:70


W: unbound identifier 'x' at <unknown>:238:22
W: unbound identifier 'x' at <unknown>:238:68
W: unbound identifier 'x' at <unknown>:260:22
W: unbound identifier 'x' at <unknown>:260:68
W: unbound identifier 'x' at <unknown>:238:22
W: unbound identifier 'x' at <unknown>:238:68
W: unbound identifier 'x' at <unknown>:260:22
W: unbound identifier 'x' at <unknown>:260:68
W: unbound identifier 'x' at <unknown>:88:22
W: unbound identifier 'x' at <unknown>:88:68
W: unbound identifier 'x' at <unknown>:431:22
W: unbound identifier 'x' at <unknown>:431:68
W: unbound identifier 'x' at <unknown>:445:22
W: unbound identifier 'x' at <unknown>:445:68
W: unbound identifier 'x' at <unknown>:88:22
W: unbound identifier 'x' at <unknown>:88:68
W: unbound identifier 'x' at <unknown>:431:22
W: unbound identifier 'x' at <unknown>:431:68
W: unbound identifier 'x' at <unknown>:445:22
W: unbound identifier 'x' at <unknown>:445:68


W: unbound identifier 'x' at <unknown>:263:22
W: unbound identifier 'x' at <unknown>:263:68
W: unbound identifier 'x' at <unknown>:263:22
W: unbound identifier 'x' at <unknown>:263:68
W: unbound identifier 'x' at <unknown>:181:22
W: unbound identifier 'x' at <unknown>:181:68
W: unbound identifier 'x' at <unknown>:182:22
W: unbound identifier 'x' at <unknown>:182:68


[
  {
    "file": "ASYNC.py",
    "scope_type": "Module",
    "scope_name": "<module>",
    "scope_lineno": null,
    "def_name": "client_ASYNC",
    "def_node_type": "alias",
    "def_lineno": 2,
    "def_islive": true,
    "user_count": 0,
    "is_unused": true
  },
  {
    "file": "ASYNC.py",
    "scope_type": "Module",
    "scope_name": "<module>",
    "scope_lineno": null,
    "def_name": "Client",
    "def_node_type": "alias",
    "def_lineno": 4,
    "def_islive": true,
    "user_count": 0,
    "is_unused": true
  },
  {
    "file": "ASYNC.py",
    "scope_type": "Module",
    "scope_name": "<module>",
    "scope_lineno": null,
    "def_name": "RedditClient",
    "def_node_type": "alias",
    "def_lineno": 5,
    "def_islive": true,
    "user_count": 0,
    "is_unused": true
  },
  {
    "file": "ASYNC.py",
    "scope_type": "Module",
    "scope_name": "<module>",
    "scope_lineno": null,
    "def_name": "Reddit",
    "def_node_type": "alias",
    "def_lineno": 6,
    "def_isliv